# Project 2: Text Similarity & Duplicate Detection (G05)

## Notebook 2 (Approach 2): Pretrained Sentence Transformers + Classifier

**Goal:** Upgrade feature extraction by using dense vector embeddings to predict whether two questions are duplicates.

**Why this approach?**
* It introduces dense vector embeddings while still keeping the classification step simple.
* Sentence Transformers successfully capture the actual meaning and synonyms behind words, rather than relying strictly on exact lexical overlap like TF-IDF.

**Pipeline overview**
* Load the Quora Question Pairs dataset and initialize the pretrained sentence transformer (`all-MiniLM-L6-v2`).
* Encode each question in the pair independently into a high-dimensional vector using a "Bi-Encoder" architecture.
* Calculate the absolute difference between the two question embeddings to create the aggregated feature.
* Train a Logistic Regression model on this difference vector to classify duplicates.
* Evaluate the final results.

Step 1: Setup and Imports

In [2]:
!pip install sentence-transformers pandas scikit-learn

In [1]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

Step 2: Load Data and Pretrained Model

In [7]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer

print("1. Loading dataset from local CSV file...")

data_path = "/content/sample_data/questions.csv"
if not os.path.exists(data_path):
    raise FileNotFoundError(
        f"Could not find {data_path!r} in the current folder. "
    )

df = pd.read_csv(data_path)

# Keep only the columns we actually need for the model
required_cols = ["question1", "question2", "is_duplicate"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}. Found: {list(df.columns)[:20]}...")

df = df[required_cols].dropna()

# Sample 20,000 rows to save RAM and time
# df = df.sample(n=20000, random_state=42)

print(f"-> Dataset loaded successfully with {df.shape[0]} rows!\n")

# Load the pretrained sentence transformer
model_st = SentenceTransformer('all-MiniLM-L6-v2')

1. Loading dataset from local CSV file...
-> Dataset loaded successfully with 404348 rows!



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step 3: Feature Extraction (Embeddings)

In [8]:
#force string type
# Step 3: Encode in Batches to save memory
embeddings1 = model_st.encode(df['question1'].astype(str).tolist(), batch_size=32, show_progress_bar=True)
embeddings2 = model_st.encode(df['question2'].astype(str).tolist(), batch_size=32, show_progress_bar=True)

Batches:   0%|          | 0/12636 [00:00<?, ?it/s]

Batches:   0%|          | 0/12636 [00:00<?, ?it/s]

Step 4: Aggregating Representations

In [9]:
import gc

# Calculate the absolute difference between the two sentence embeddings
features = np.abs(embeddings1 - embeddings2)

# Clear RAM (Garbage Collection)
del embeddings1
del embeddings2
gc.collect()

# Extract labels and split the data
labels = df['is_duplicate'].values
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

Step 5: Model Training and Evaluation

In [10]:
# Train a Logistic Regression classifier on the embedding differences
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# batch size that computer's RAM can handle
batch_size = 5000
y_pred = []

print("Predicting in batches to save memory...")

# Loop through X_test in chunks
for i in range(0, len(X_test), batch_size):
    X_batch = X_test[i : i + batch_size]
    batch_pred = clf.predict(X_batch)
    y_pred.extend(batch_pred)

y_pred = np.array(y_pred)

# Predict and evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Predicting in batches to save memory...
Accuracy: 0.7927290713490788

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.84      0.84     51240
           1       0.72      0.70      0.71     29630

    accuracy                           0.79     80870
   macro avg       0.78      0.77      0.78     80870
weighted avg       0.79      0.79      0.79     80870

